In [1]:
%pip install "transformers<5" sacremoses sentencepiece
%pip install --index-url https://download.pytorch.org/whl/cpu torch

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


### Language Translation using Hugging Face Encoder-Decoder Model

First, let's import the necessary classes from the `transformers` library: `AutoTokenizer` and `AutoModelForSeq2SeqLM`.

Next, we'll choose a pre-trained model. For English to French translation, `Helsinki-NLP/opus-mt-en-fr` is a good general-purpose model from the MarianMT family. We'll load both the tokenizer and the model.

Now, let's prepare some text in English and translate it to French. We'll tokenize the input, generate the translation using the model, and then decode the generated IDs back into human-readable text.

In [2]:
import sentencepiece  # Ensure the Marian tokenizer dependency is loaded in this kernel
from transformers.models.marian.tokenization_marian import MarianTokenizer
from transformers import AutoModelForSeq2SeqLM

# Specify the model name (English to French translation example)
model_name = "Helsinki-NLP/opus-mt-en-fr"

# Load the tokenizer
tokenizer = MarianTokenizer.from_pretrained(model_name)

# Load the model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print(f"Model '{model_name}' and tokenizer loaded successfully.")

# Text to be translated
english_text = "Hello, how are you today? This is a beautiful day."

# Tokenize the input text
inputs = tokenizer(english_text, return_tensors="pt")

# Generate the translation
# The 'max_length' parameter prevents infinite loops for some models
# The 'num_beams' parameter is for beam search decoding, which often yields better translations
# The 'early_stopping' parameter stops the generation when all beam hypotheses have finished
translated_tokens = model.generate(**inputs, max_length=50, num_beams=5, early_stopping=True)

# Decode the translated tokens back to text
french_text = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

print(f"Original English Text: {english_text}")
print(f"Translated French Text: {french_text}")

/home/dhruv/sem5/AI/Lab/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model 'Helsinki-NLP/opus-mt-en-fr' and tokenizer loaded successfully.
Original English Text: Hello, how are you today? This is a beautiful day.
Translated French Text: Comment allez-vous aujourd'hui ?


### Next Token Prediction / Sentence Completion

This section demonstrates how to use a pre-trained causal language model (like GPT-2) from Hugging Face to predict the next token(s) and complete a given sentence. We use the `pipeline` abstraction, which simplifies the process of using models for common tasks like text generation.

- **`pipeline('text-generation', model='gpt2')`**: Initializes a text generation pipeline using the GPT-2 model. This pipeline handles tokenization, model inference, and decoding automatically.
- **`start_sentence`**: The initial text provided to the model.
- **`generator(...)`**: Calls the text generation pipeline with the input sentence and specifies parameters:
    - `max_new_tokens`: The maximum number of new tokens the model should generate to complete the sentence.
    - `num_return_sequences`: How many different possible completions the model should provide.
    - `clean_up_tokenization_spaces`: Helps in cleaning up any extra spaces introduced during tokenization for better output readability.

In [3]:
from transformers import pipeline

# Load a text generation pipeline with a causal language model
# 'gpt2' is a common example for sentence completion
generator = pipeline('text-generation', model='gpt2', device=-1)

# Define the starting sentence
start_sentence = "The quick brown fox jumps over the"

# Generate text to complete the sentence
# max_new_tokens: maximum number of new tokens to generate
# num_return_sequences: number of different completions to generate
# clean_up_tokenization_spaces: removes extra spaces from tokenization

# You can adjust max_new_tokens and num_return_sequences as needed.
# Higher num_beams can lead to better quality but slower generation.

print(f"Starting sentence: {start_sentence}")

results = generator(start_sentence, max_new_tokens=20, num_return_sequences=1, clean_up_tokenization_spaces=True)

for i, result in enumerate(results):
    print(f"\nGenerated Completion {i+1}:")
    print(result['generated_text'])


Device set to use cpu
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Starting sentence: The quick brown fox jumps over the

Generated Completion 1:
The quick brown fox jumps over the front of the car, but the rabbit's eyes are still closed. The fox does not move and


### Sentiment Analysis

Sentiment analysis (also known as opinion mining) is a natural language processing (NLP) technique used to determine whether data is positive, negative, or neutral. Here, we'll use a pre-trained Hugging Face model designed for sentiment analysis to classify text.

In [3]:
from transformers import pipeline

# Use a pretrained sentiment-classification model and force CPU execution.
sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=-1,
)

texts_to_analyze = [
    "This movie was fantastic! A truly enjoyable experience.",
    "The service was slow and disappointing.",
]

results = sentiment_analyzer(texts_to_analyze)
for text, result in zip(texts_to_analyze, results):
    print(f"Text: {text}")
    print(f"Sentiment: {result['label']} (score: {result['score']:.4f})\n")

Device set to use cpu


Text: This movie was fantastic! A truly enjoyable experience.
Sentiment: POSITIVE (score: 0.9999)

Text: The service was slow and disappointing.
Sentiment: NEGATIVE (score: 0.9997)

